# 04 — Multi-target model: ESM-2 protein embedding + GINE molecule graph

Phase 4 of **CRC_Inhibitor_ML**. The headline architectural result — one multi-modal model that handles all four CRC targets via a protein-sequence input.

## What's new vs Phase 3

- **Two inputs per row** — instead of "molecule → pIC50," the model sees "molecule + target protein → pIC50"
- **Protein representation** — each target's amino acid sequence runs through **ESM-2** (Meta's protein language model) once, producing a fixed-size embedding that captures structural/functional features of the protein. Cached and reused at training time.
- **One model, four targets** — instead of 4 separate single-target GNNs, one network with shared parameters. KRAS (small dataset, 2.6K molecules) benefits from the gradient signal of EGFR (10K molecules) because the molecule-graph branch is shared.
- **Generalizable** — in principle, this model could be evaluated on a 5th, 6th, ... Nth target it never saw during training, just by feeding that target's ESM-2 embedding alongside candidate molecules. Phase 5 (the CLI tool) exposes this.

## Architecture

```
Molecule (SMILES → PyG graph)            Target (UniProt sequence)
        │                                          │
        ▼                                          ▼
  3 × GINE layers + BatchNorm           ESM-2 (frozen, pre-cached)
        │                                          │
        ▼                                          ▼
  Sum-pooled molecule embedding         Mean-pooled protein embedding
        │  (128-dim)                              │  (480-dim)
        │                                          │
        │                                          ▼
        │                                  Linear → 128-dim
        │                                          │
        └──────────────┬───────────────────────────┘
                       │
                       ▼
                  concat (256-dim)
                       │
                       ▼
                MLP predictor → predicted pIC50
```

Same three-way scaffold split as Phase 3.5, applied per-target then pooled — so each target contributes balanced samples to train/val/test.

In [ ]:
from pathlib import Path
from collections import defaultdict
import io
import json
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold

from transformers import AutoTokenizer, AutoModel
from Bio import SeqIO

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RDLogger.DisableLog("rdApp.*")

PROJECT_ROOT  = Path(r"C:\Users\palla\OneDrive\Documents\Coding Projects\CRC_Inhibitor_ML")
CLEAN_CSV     = PROJECT_ROOT / "data" / "interim"   / "chembl_crc_targets_clean.csv"
MAPPING_TXT   = PROJECT_ROOT / "data" / "raw"       / "chembl_uniprot_mapping.txt"
MODELS_DIR    = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR   = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ESM_CACHE     = PROCESSED_DIR / "target_esm2_embeddings.pt"
PHASE4_METRICS_PATH = REPORTS_DIR / "phase4_multi_target_metrics.json"
RF_METRICS_PATH     = REPORTS_DIR / "baseline_rf_metrics.json"
GNN_V1_METRICS_PATH = REPORTS_DIR / "gnn_single_target_metrics.json"
GNN_V2_METRICS_PATH = REPORTS_DIR / "gnn_v2_metrics.json"

TARGETS = {
    "CHEMBL2189121": "KRAS",
    "CHEMBL5145":    "BRAF",
    "CHEMBL203":     "EGFR",
    "CHEMBL4005":    "PIK3CA",
}

ESM_MODEL_NAME = "facebook/esm2_t12_35M_UR50D"  # small ESM-2: ~150 MB, 480-dim embedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## Cell 1 — look up UniProt accessions for our targets

ChEMBL IDs (e.g. CHEMBL203) map to UniProt accessions (e.g. P00533 for EGFR) via the `chembl_uniprot_mapping.txt` file we downloaded in Phase 0. Demonstrates the lookup mechanism the CLI tool (Phase 5) will use.

In [ ]:
# The mapping file is tab-separated: uniprot_acc | chembl_target_id | name | type
# Comments start with '#'
mapping = pd.read_csv(
    MAPPING_TXT, sep="\t", comment="#", header=None,
    names=["uniprot", "target_chembl_id", "name", "target_type"],
)
print(f"Loaded {len(mapping):,} ChEMBL↔UniProt mappings")

# For each of our 4 targets, grab the first matching UniProt accession
target_to_uniprot = {}
for chembl_id, name in TARGETS.items():
    matches = mapping[mapping["target_chembl_id"] == chembl_id]
    if len(matches) == 0:
        raise ValueError(f"No UniProt mapping found for {chembl_id} ({name})")
    target_to_uniprot[chembl_id] = matches.iloc[0]["uniprot"]
    print(f"  {name:7s} ({chembl_id})  →  UniProt {matches.iloc[0]['uniprot']}")

## Cell 2 — fetch protein sequences from UniProt

One small HTTP request per target → the canonical amino acid sequence in FASTA format. Parse with biopython.

These are short (KRAS ~189 residues, EGFR ~1210, BRAF ~766, PIK3CA ~1068) and free to fetch from the public UniProt REST API.

In [ ]:
def fetch_uniprot_sequence(accession):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    record = next(SeqIO.parse(io.StringIO(response.text), "fasta"))
    return str(record.seq)

target_to_sequence = {}
for chembl_id, name in TARGETS.items():
    seq = fetch_uniprot_sequence(target_to_uniprot[chembl_id])
    target_to_sequence[chembl_id] = seq
    print(f"  {name:7s}  {len(seq):>5d} residues  starts: {seq[:30]}...")

## Cell 3 — load ESM-2 and embed the 4 target sequences

**ESM-2** is Meta's protein language model — a transformer trained on hundreds of millions of protein sequences. For any sequence input, it produces a per-residue 480-dim representation (for the small `t12_35M` variant). We pool over residues to get a single 480-dim vector per protein.

Why this works: ESM-2 was trained with masked-language-model objective on protein sequences, so its embeddings encode structural/functional features (active sites, folds, family membership) that are highly predictive of binding behavior — without us having to compute structures ourselves.

We run this **once** for the 4 targets, cache the embeddings to `data/processed/target_esm2_embeddings.pt`, and reuse forever. No need for ESM-2 to be active during training — it'd be wasted GPU memory.

In [ ]:
if ESM_CACHE.exists():
    target_embeddings = torch.load(ESM_CACHE, map_location="cpu")
    print(f"Loaded cached ESM-2 embeddings from {ESM_CACHE.name}")
else:
    print(f"Downloading + loading ESM-2: {ESM_MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(ESM_MODEL_NAME)
    esm_model = AutoModel.from_pretrained(ESM_MODEL_NAME).to(device).eval()

    target_embeddings = {}
    for chembl_id, name in TARGETS.items():
        seq = target_to_sequence[chembl_id]
        with torch.no_grad():
            inputs = tokenizer(seq, return_tensors="pt").to(device)
            outputs = esm_model(**inputs)
            # Mean-pool over residues (skip CLS at position 0 and EOS at end)
            emb = outputs.last_hidden_state[0, 1:-1, :].mean(dim=0).cpu()
        target_embeddings[chembl_id] = emb
        print(f"  {name:7s}  embedding shape: {tuple(emb.shape)}  norm: {emb.norm().item():.2f}")

    torch.save(target_embeddings, ESM_CACHE)
    print(f"\nCached to {ESM_CACHE.name}")
    # Free ESM-2 from memory — we don't need it during training
    del esm_model, tokenizer
    if device.type == "cuda":
        torch.cuda.empty_cache()

ESM_DIM = next(iter(target_embeddings.values())).shape[0]
print(f"\nESM-2 embedding dim: {ESM_DIM}")

## Cell 4 — load clean molecules + featurize as PyG graphs (with target embedding attached)

Same SMILES featurization as Phase 3, but each `Data` object now also carries its target's ESM-2 embedding as `target_emb`. When PyG batches these, the batch will have a `target_emb` tensor of shape `(batch_size, ESM_DIM)` — one row per molecule in the batch, pointing to its specific target.

In [ ]:
ATOM_TYPES = ["C", "N", "O", "F", "P", "S", "Cl", "Br", "I", "B", "Si", "H", "Other"]
HYB_TYPES = [
    Chem.HybridizationType.S, Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
    Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D, Chem.HybridizationType.SP3D2,
]
BOND_TYPES = [
    Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC,
]
N_ATOM_FEATURES = len(ATOM_TYPES) + len(HYB_TYPES) + 5
N_BOND_FEATURES = len(BOND_TYPES) + 2

def one_hot(value, options):
    return [int(value == o) for o in options]

def atom_features(atom):
    sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES else "Other"
    return (
        one_hot(sym, ATOM_TYPES)
        + one_hot(atom.GetHybridization(), HYB_TYPES)
        + [atom.GetFormalCharge(), atom.GetTotalNumHs(), atom.GetDegree(),
           int(atom.GetIsAromatic()), int(atom.IsInRing())]
    )

def bond_features(bond):
    return one_hot(bond.GetBondType(), BOND_TYPES) + [int(bond.GetIsConjugated()), int(bond.IsInRing())]

def mol_to_pyg(mol, y_value, target_emb):
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    src, dst, eattr = [], [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        src += [i, j]; dst += [j, i]; eattr += [bf, bf]
    if len(src) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, N_BOND_FEATURES), dtype=torch.float)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr  = torch.tensor(eattr, dtype=torch.float)
    return Data(
        x=x, edge_index=edge_index, edge_attr=edge_attr,
        y=torch.tensor([y_value], dtype=torch.float),
        target_emb=target_emb.unsqueeze(0),   # shape (1, ESM_DIM) so PyG batches it correctly
    )

# Load + parse + scaffold once
print("Loading clean data...")
clean = pd.read_csv(CLEAN_CSV)

print("Parsing SMILES + computing scaffolds...")
mols = [Chem.MolFromSmiles(s) for s in tqdm(clean["canonical_smiles_std"])]
valid = np.array([m is not None for m in mols])
clean = clean[valid].reset_index(drop=True)
mols  = [m for m, ok in zip(mols, valid) if ok]

scaffolds = []
for m in tqdm(mols):
    try:
        scaffolds.append(MurckoScaffold.MurckoScaffoldSmiles(mol=m, includeChirality=False))
    except Exception:
        scaffolds.append("")
clean["scaffold"] = scaffolds

print("\nFeaturizing molecules (with target embeddings attached)...")
pyg_data = []
for mol, y, tid in tqdm(
    list(zip(mols, clean["pic50"].values, clean["target_chembl_id"].values)),
    total=len(mols),
):
    pyg_data.append(mol_to_pyg(mol, float(y), target_embeddings[tid]))
print(f"\nBuilt {len(pyg_data):,} graphs, each carrying its target's ESM-2 embedding")

## Cell 5 — three-way scaffold split (per-target, then concatenated)

Run the same three-way scaffold split as Phase 3.5 separately for each target, then pool the results. This guarantees:

- Each target contributes proportional molecules to train, val, and test
- Scaffolds are never shared across splits within a target
- The model trains on a mix of all 4 targets simultaneously

In [ ]:
def three_way_scaffold_split(scaffolds, frac_train=0.7, frac_val=0.15):
    groups = defaultdict(list)
    for i, s in enumerate(scaffolds):
        groups[s].append(i)
    sorted_groups = sorted(groups.values(), key=len, reverse=True)
    n_total = len(scaffolds)
    train_cutoff = int(frac_train * n_total)
    val_cutoff   = train_cutoff + int(frac_val * n_total)
    train, val, test = [], [], []
    for indices in sorted_groups:
        if len(train) + len(indices) <= train_cutoff:
            train.extend(indices)
        elif len(train) + len(val) + len(indices) <= val_cutoff:
            val.extend(indices)
        else:
            test.extend(indices)
    return np.array(train), np.array(val), np.array(test)

# Per-target split, then pool
all_train_idx, all_val_idx, all_test_idx = [], [], []
for target_id, target_name in TARGETS.items():
    mask = (clean["target_chembl_id"] == target_id).values
    target_indices = np.where(mask)[0]
    target_scaffolds = clean.loc[mask, "scaffold"].tolist()
    tr_local, va_local, te_local = three_way_scaffold_split(target_scaffolds)
    all_train_idx.extend(target_indices[tr_local])
    all_val_idx.extend(target_indices[va_local])
    all_test_idx.extend(target_indices[te_local])
    print(f"  {target_name:7s}  train={len(tr_local):>5,}  val={len(va_local):>5,}  test={len(te_local):>5,}")

all_train_idx = np.array(all_train_idx)
all_val_idx   = np.array(all_val_idx)
all_test_idx  = np.array(all_test_idx)

print(f"\nPOOLED: train={len(all_train_idx):,}  val={len(all_val_idx):,}  test={len(all_test_idx):,}")

## Cell 6 — multi-modal GINE model

Same molecule branch as Phase 3.5 (3 × GINEConv + BatchNorm), plus a target branch that projects the 480-dim ESM-2 embedding to 128-dim, then concatenates with the pooled molecule vector. Predictor head sees a 256-dim joint representation.

In [ ]:
class MultiModalGINE(nn.Module):
    def __init__(self, atom_dim, edge_dim, target_dim, hidden_dim=128, n_layers=3, dropout=0.2):
        super().__init__()
        self.input_proj  = nn.Linear(atom_dim, hidden_dim)
        self.edge_proj   = nn.Linear(edge_dim, hidden_dim)
        self.target_proj = nn.Sequential(
            nn.Linear(target_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(mlp))
            self.bns.append(nn.BatchNorm1d(hidden_dim))

        self.dropout = nn.Dropout(dropout)
        self.predictor = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        target_emb = data.target_emb  # (batch_size, target_dim)

        # Molecule branch
        h = self.input_proj(x)
        e = self.edge_proj(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index, e)
            h = bn(h)
            h = F.relu(h)
            h = self.dropout(h)
        mol_emb = global_add_pool(h, batch)  # (batch_size, hidden_dim)

        # Target branch
        tgt_emb = self.target_proj(target_emb)  # (batch_size, hidden_dim)

        # Fuse and predict
        combined = torch.cat([mol_emb, tgt_emb], dim=1)  # (batch_size, 2*hidden_dim)
        return self.predictor(combined).squeeze(-1)

## Cell 7 — train the single multi-target model

One training run, not four. Same hyperparameters as Phase 3.5 (Adam, lr=1e-3, batch 64, dropout 0.2, ReduceLROnPlateau, patience 25, max 200 epochs). The model sees mixed batches with molecules from all 4 targets and learns to discriminate by the ESM-2 embedding.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    preds_all, y_all = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch).cpu().numpy()
            preds_all.append(pred)
            y_all.append(batch.y.cpu().numpy())
    return np.concatenate(preds_all), np.concatenate(y_all)

train_data = [pyg_data[i] for i in all_train_idx]
val_data   = [pyg_data[i] for i in all_val_idx]
test_data  = [pyg_data[i] for i in all_test_idx]

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

model = MultiModalGINE(
    atom_dim=N_ATOM_FEATURES,
    edge_dim=N_BOND_FEATURES,
    target_dim=ESM_DIM,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
loss_fn = nn.MSELoss()

best_val_rmse = float("inf")
best_state    = None
epochs_no_improve = 0
PATIENCE = 25
EPOCHS   = 200

pbar = tqdm(range(EPOCHS), desc="multi-target")
for epoch in pbar:
    model.train()
    running = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred, batch.y)
        loss.backward()
        optimizer.step()
        running += loss.item() * batch.num_graphs
    train_loss = running / len(train_data)

    vp, vy = evaluate_model(model, val_loader)
    val_rmse = float(np.sqrt(mean_squared_error(vy, vp)))
    scheduler.step(val_rmse)

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    pbar.set_postfix({
        "train_mse": f"{train_loss:.3f}",
        "val_rmse":  f"{val_rmse:.3f}",
        "best_val":  f"{best_val_rmse:.3f}",
        "lr":        f"{optimizer.param_groups[0]['lr']:.1e}",
    })

    if epochs_no_improve >= PATIENCE:
        break

model.load_state_dict(best_state)
print(f"\nTrained {epoch + 1} epochs, best val RMSE = {best_val_rmse:.3f}")

## Cell 8 — per-target test evaluation

Run the trained multi-target model on the test set, then break out metrics per target so we can compare against the Phase 2 / 3 single-target results. **This is the key apples-to-apples comparison.**

In [ ]:
test_preds, test_y = evaluate_model(model, test_loader)
test_target_ids = [clean.iloc[i]["target_chembl_id"] for i in all_test_idx]

phase4_results = []
print("Per-target test metrics:")
for target_id, target_name in TARGETS.items():
    mask = np.array([tid == target_id for tid in test_target_ids])
    if mask.sum() == 0:
        continue
    y_t  = test_y[mask]
    p_t  = test_preds[mask]
    r2   = float(r2_score(y_t, p_t))
    rmse = float(np.sqrt(mean_squared_error(y_t, p_t)))
    mae  = float(mean_absolute_error(y_t, p_t))
    phase4_results.append({
        "target": target_name,
        "n_test": int(mask.sum()),
        "r2":     r2,
        "rmse":   rmse,
        "mae":    mae,
    })
    print(f"  {target_name:7s}  n={mask.sum():>5d}  R\u00b2={r2:6.3f}  RMSE={rmse:.3f}  MAE={mae:.3f}")

# Overall (pooled across all 4 targets)
overall_r2 = float(r2_score(test_y, test_preds))
overall_rmse = float(np.sqrt(mean_squared_error(test_y, test_preds)))
print(f"\nPOOLED  n={len(test_y):>5d}  R\u00b2={overall_r2:6.3f}  RMSE={overall_rmse:.3f}")

## Cell 9 — full four-way comparison

Side-by-side scaffold-split R² for: RF baseline, single-target GIN (Phase 3.0), single-target GINE (Phase 3.5), and multi-target ESM-2 (Phase 4).

In [ ]:
def load_metrics(path, which="r2"):
    with open(path) as f:
        d = json.load(f)
    return {r["target"]: r[which] for r in d if r.get("split", "scaffold") == "scaffold"}

rf_r2     = load_metrics(RF_METRICS_PATH)
gnn_v1_r2 = {r["target"]: r["r2"] for r in json.load(open(GNN_V1_METRICS_PATH))}
gnn_v2_r2 = {r["target"]: r["r2"] for r in json.load(open(GNN_V2_METRICS_PATH))}
phase4_r2 = {r["target"]: r["r2"] for r in phase4_results}

compare = pd.DataFrame({
    "RF baseline":        rf_r2,
    "GIN single (3.0)":   gnn_v1_r2,
    "GINE single (3.5)":  gnn_v2_r2,
    "GINE+ESM-2 (4.0)":   phase4_r2,
}).reindex(index=["KRAS", "BRAF", "EGFR", "PIK3CA"])
compare["delta_p4_vs_rf"] = compare["GINE+ESM-2 (4.0)"] - compare["RF baseline"]

print("Scaffold-split test R² (higher = better):\n")
print(compare.round(3))

## Cell 10 — save metrics + checkpoint

In [ ]:
with open(PHASE4_METRICS_PATH, "w") as f:
    json.dump(phase4_results, f, indent=2)

ckpt_path = MODELS_DIR / "gine_esm2_multi_target.pt"
torch.save({
    "state_dict":       model.state_dict(),
    "atom_dim":         N_ATOM_FEATURES,
    "edge_dim":         N_BOND_FEATURES,
    "target_dim":       ESM_DIM,
    "esm_model_name":   ESM_MODEL_NAME,
    "targets":          TARGETS,
    "target_to_uniprot": target_to_uniprot,
    "per_target_metrics": phase4_results,
    "best_val_rmse":    best_val_rmse,
}, ckpt_path)

print(f"Saved Phase 4 metrics  \u2192 {PHASE4_METRICS_PATH.name}")
print(f"Saved Phase 4 model    \u2192 {ckpt_path.name}")
print()
print(json.dumps(phase4_results, indent=2))